# Dự án 1: Phân tích kinh tế vĩ mô và thị trường tài chính

## Mục tiêu
Dự án này tổng hợp lại các kỹ thuật bạn đã học về **merge**, **merge ordered**,
xử lý dữ liệu theo thời gian và trực quan hóa. Bộ notebook tập trung vào 3 câu hỏi:

1. GDP của 4 nền kinh tế lớn (Mỹ, Trung Quốc, Đức, Nhật) thay đổi như thế nào trong giai đoạn 2010-2018?
2. Sau khi ghép GDP với dân số, GDP bình quân đầu người khác nhau ra sao giữa các nước?
3. Với riêng Mỹ, tăng trưởng GDP có đi cùng chiều với lợi suất S&P500 hay không?

## Dữ liệu sử dụng
- `WorldBank_GDP.csv`
- `WorldBank_POP.csv`
- `S&P500.csv`

## Lưu ý về dữ liệu
- File dân số có **bản ghi trùng ở năm 2012**, nên notebook sẽ làm sạch trước khi phân tích.
- Dữ liệu S&P500 hiện chỉ đủ cho giai đoạn 2008-2017 và số quan sát không nhiều, vì vậy phần tương quan chỉ mang tính **khám phá**.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlelocation'] = 'left'

DATA_DIR = Path('data')

gdp = pd.read_csv(DATA_DIR / 'WorldBank_GDP.csv')
pop = pd.read_csv(DATA_DIR / 'WorldBank_POP.csv')
sp500 = pd.read_csv(DATA_DIR / 'S&P500.csv').rename(columns={'Date': 'Year', 'Returns': 'SP500_Returns'})

print('Kích thước GDP   :', gdp.shape)
print('Kích thước POP   :', pop.shape)
print('Kích thước S&P500:', sp500.shape)
gdp.head()

## 1. Làm sạch và hợp nhất dữ liệu

Bước quan trọng nhất của dự án là kiểm tra chất lượng dữ liệu trước khi merge.
Ở đây, ta sẽ:

- kiểm tra khóa ghép `Country Code` + `Year`
- loại trùng ở bảng dân số
- ghép GDP với dân số để tạo thêm biến **GDP per capita**

In [ ]:
gdp_duplicate_keys = (
    gdp.groupby(['Country Code', 'Year'])
       .size()
       .reset_index(name='n')
       .query('n > 1')
)
pop_duplicate_keys = (
    pop.groupby(['Country Code', 'Year'])
       .size()
       .reset_index(name='n')
       .query('n > 1')
)

print('Số khóa bị trùng trong bảng GDP     :', len(gdp_duplicate_keys))
print('Số khóa bị trùng trong bảng dân số  :', len(pop_duplicate_keys))
print('Ví dụ khóa bị trùng ở GDP:')
display(gdp_duplicate_keys.head())
print('Ví dụ khóa bị trùng ở bảng dân số:')
display(pop_duplicate_keys.head())

gdp_clean = gdp.drop_duplicates(subset=['Country Code', 'Year']).copy()
pop_clean = pop.drop_duplicates(subset=['Country Code', 'Year']).copy()
sp500 = sp500.sort_values('Year').reset_index(drop=True)

macro = (
    gdp_clean.merge(
        pop_clean[['Country Code', 'Country Name', 'Year', 'Pop']],
        on=['Country Code', 'Country Name', 'Year'],
        how='left'
    )
    .sort_values(['Country Name', 'Year'])
    .reset_index(drop=True)
)
macro['GDP_per_capita'] = macro['GDP'] / macro['Pop']
macro['GDP_growth_pct'] = macro.groupby('Country Code')['GDP'].pct_change() * 100
macro['Pop_growth_pct'] = macro.groupby('Country Code')['Pop'].pct_change() * 100
macro['GDP_per_capita_growth_pct'] = macro.groupby('Country Code')['GDP_per_capita'].pct_change() * 100

print('Số giá trị thiếu sau khi ghép:')
display(macro.isna().sum().to_frame('missing'))
macro.head(8)

## 2. Xu hướng GDP theo quốc gia

In [ ]:
pivot_gdp = macro.pivot(index='Year', columns='Country Name', values='GDP') / 1e12
ax = pivot_gdp.plot(marker='o')
ax.set_title('GDP trend by country (trillion USD)')
ax.set_xlabel('Year')
ax.set_ylabel('GDP (trillion USD)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 3. GDP bình quân đầu người

GDP tuyệt đối giúp nhìn quy mô nền kinh tế, nhưng GDP bình quân đầu người cho thấy
mức sản lượng tạo ra bình quân trên mỗi người dân.

In [ ]:
pivot_pc = macro.pivot(index='Year', columns='Country Name', values='GDP_per_capita')
ax = pivot_pc.plot(marker='o')
ax.set_title('GDP per capita trend by country (USD)')
ax.set_xlabel('Year')
ax.set_ylabel('GDP per capita (USD)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Cơ cấu GDP năm 2018 trong nhóm 4 nước

In [ ]:
gdp_2018 = macro.loc[macro['Year'] == 2018, ['Country Name', 'GDP']].copy()
gdp_2018['GDP_share_pct'] = gdp_2018['GDP'] / gdp_2018['GDP'].sum() * 100
display(gdp_2018.sort_values('GDP_share_pct', ascending=False))

ax = gdp_2018.sort_values('GDP_share_pct', ascending=False).plot(
    kind='bar', x='Country Name', y='GDP_share_pct', legend=False
)
ax.set_title('GDP share in 2018 within the 4-country sample')
ax.set_xlabel('Country')
ax.set_ylabel('Share (%)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 5. Mỹ: tăng trưởng GDP và lợi suất S&P500

Vì S&P500 phản ánh thị trường chứng khoán Mỹ, phần dưới đây chỉ giữ lại dữ liệu của Mỹ
rồi ghép với lợi suất S&P500 theo năm.

In [ ]:
us = (
    macro.loc[macro['Country Code'] == 'USA', ['Year', 'GDP', 'GDP_per_capita', 'GDP_growth_pct', 'GDP_per_capita_growth_pct']]
         .merge(sp500, on='Year', how='inner')
         .sort_values('Year')
         .reset_index(drop=True)
)

display(us)
display(us[['GDP_growth_pct', 'GDP_per_capita_growth_pct', 'SP500_Returns']].corr())

In [ ]:
ax = us.plot(x='Year', y='GDP_growth_pct', marker='o', legend=False)
ax.set_title('US GDP growth rate by year (%)')
ax.set_xlabel('Year')
ax.set_ylabel('GDP growth (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
ax = us.plot(kind='scatter', x='GDP_growth_pct', y='SP500_Returns')
ax.set_title('US GDP growth vs S&P500 returns')
ax.set_xlabel('GDP growth (%)')
ax.set_ylabel('S&P500 returns (%)')
plt.tight_layout()
plt.show()

## 6. Kết luận ngắn

In [ ]:
growth_2010_2018 = (
    macro.pivot(index='Country Name', columns='Year', values='GDP_per_capita')
         .assign(total_growth_pct=lambda d: (d[2018] / d[2010] - 1) * 100)
         [['total_growth_pct']]
         .sort_values('total_growth_pct', ascending=False)
)

top_pc_2018 = macro.loc[macro['Year'] == 2018, ['Country Name', 'GDP_per_capita']].sort_values('GDP_per_capita', ascending=False)
corr_value = us[['GDP_growth_pct', 'SP500_Returns']].corr().iloc[0, 1]

print('1) Mỹ vẫn là nền kinh tế lớn nhất trong mẫu 4 nước ở năm 2018.')
print('2) Xét GDP bình quân đầu người năm 2018, thứ hạng là:')
display(top_pc_2018)
print('3) Tăng trưởng GDP bình quân đầu người giai đoạn 2010-2018:')
display(growth_2010_2018)
print(f'4) Tương quan giữa GDP growth của Mỹ và S&P500 returns trong mẫu nhỏ này: {corr_value:.3f}')
print('   -> Giá trị này khá thấp, nên chưa thể kết luận hai biến đi cùng chiều trong bộ dữ liệu hiện có.')